# 📉 Project FORESIGHT – Notebook 05: Backtesting & Risk Analysis
**NorthBay Living | Demand & Inventory Intelligence**

Performs detailed backtesting analysis and risk scoring validation.


In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from src.evaluation import compute_all_metrics, wape, mape
from src.risk_scoring import run_risk_scoring, calculate_business_impact

DATA   = Path('../data/processed')
MODELS = Path('../models')
LAYOUT = dict(paper_bgcolor='#1e293b', plot_bgcolor='#1e293b', font=dict(color='#94a3b8'))
print('Ready ✓')

## 1. Load Results

In [ ]:
comp_file = DATA / 'model_comparison.csv'
if comp_file.exists():
    comp = pd.read_csv(comp_file)
    print('Model Comparison Results:')
    print(comp.to_string(index=False))
else:
    print('Run pipeline first: python main.py')

risk_file = DATA / 'risk_scores.csv'
if risk_file.exists():
    risk = pd.read_csv(risk_file)
    print(f'\nRisk scores: {len(risk):,} SKUs')

## 2. Backtesting: Error Distribution

In [ ]:
feat = pd.read_csv(DATA / 'features_engineered.csv', parse_dates=['date'], low_memory=False)
fcast = pd.read_csv(DATA / 'forecast_output.csv', parse_dates=['date'])

# Align overlapping dates (in-sample check)
feat_last30 = feat.sort_values('date').groupby('stock_code').tail(30)
merged = feat_last30.merge(fcast, on=['stock_code','date'], how='inner')

if len(merged) > 0 and 'forecast_quantity' in merged.columns:
    merged['error']     = merged['forecast_quantity'] - merged['quantity']
    merged['abs_error'] = merged['error'].abs()
    merged['pct_error'] = (merged['abs_error'] / merged['quantity'].clip(lower=0.1)) * 100
    
    fig = go.Figure(go.Histogram(x=merged['error'].clip(-200, 200), nbinsx=50,
        marker_color='#6366f1', opacity=0.8))
    fig.add_vline(x=0, line_dash='dash', line_color='#f59e0b')
    fig.update_layout(title='Forecast Error Distribution', height=380, **LAYOUT)
    fig.show()
    
    print(f'Mean Error (Bias): {merged["error"].mean():.3f}')
    print(f'Std Error:         {merged["error"].std():.3f}')
    print(f'MAE:               {merged["abs_error"].mean():.3f}')
else:
    print('No overlapping forecast/actual dates found for backtesting.')

## 3. SKU-Level WAPE Distribution

In [ ]:
# Compute per-SKU WAPE from validation (if available)
if comp_file.exists():
    fig = go.Figure()
    metrics_plot = ['WAPE','MAPE','MAE','RMSE']
    avail = [m for m in metrics_plot if m in comp.columns]
    
    radar_fig = go.Figure(go.Scatterpolar(
        r=[comp.iloc[0].get(m, 0) for m in avail] + [comp.iloc[0].get(avail[0], 0)],
        theta=avail + [avail[0]],
        fill='toself', fillcolor='rgba(99,102,241,0.2)',
        line=dict(color='#6366f1'), name=comp.iloc[0].get('model','')
    ))
    if len(comp) > 1:
        radar_fig.add_trace(go.Scatterpolar(
            r=[comp.iloc[1].get(m, 0) for m in avail] + [comp.iloc[1].get(avail[0], 0)],
            theta=avail + [avail[0]],
            fill='toself', fillcolor='rgba(6,182,212,0.2)',
            line=dict(color='#06b6d4'), name=comp.iloc[1].get('model','')
        ))
    radar_fig.update_layout(
        polar=dict(bgcolor='#1e293b', radialaxis=dict(color='#94a3b8'), angularaxis=dict(color='#94a3b8')),
        paper_bgcolor='#1e293b', font=dict(color='#94a3b8'),
        title='Model Metrics Radar Chart', height=450
    )
    radar_fig.show()

## 4. Risk Score Validation

In [ ]:
if risk_file.exists():
    risk = pd.read_csv(risk_file)
    
    print('Risk Level Distribution:')
    print(risk['risk_level'].value_counts().to_string())
    
    fig = go.Figure(go.Histogram(x=risk['risk_score'], nbinsx=30,
        marker_color='#6366f1', opacity=0.8))
    fig.update_layout(title='Risk Score Distribution (all SKUs)', height=360, **LAYOUT)
    fig.show()
    
    fig2 = px.scatter(risk.head(300), x='coverage_days', y='risk_score', color='risk_level',
        color_discrete_map={
            'Critical Stockout Risk':'#ef4444','High Stockout Risk':'#f97316',
            'Overstock Risk':'#8b5cf6','Healthy':'#10b981','Volatile Demand':'#06b6d4'
        }, title='Risk Score vs Inventory Coverage')
    fig2.update_layout(**LAYOUT, height=450)
    fig2.show()

## 5. Business Impact Summary

In [ ]:
import json
impact_file = DATA / 'business_impact.json'
if impact_file.exists():
    with open(impact_file) as f:
        impact = json.load(f)
    
    print('BUSINESS IMPACT SUMMARY')
    print('=' * 50)
    for k, v in impact.items():
        if isinstance(v, float):
            print(f'{k:35s}: £{v:>15,.2f}')
        else:
            print(f'{k:35s}: {v:>15,}')
    
    # Waterfall chart
    labels = ['Revenue at Risk','Stockout Recovery','Overstock Savings','Working Capital','Net Benefit']
    values = [
        -impact.get('revenue_at_risk',0),
        impact.get('potential_sales_increase',0),
        impact.get('overstock_cost_monthly',0),
        impact.get('working_capital_saved',0),
        impact.get('expected_profit_improvement',0),
    ]
    measures = ['absolute','relative','relative','relative','total']
    
    fig = go.Figure(go.Waterfall(
        x=labels, y=values, measure=measures,
        text=[f'£{abs(v):,.0f}' for v in values],
        textposition='outside', textfont_color='white',
        connector=dict(line=dict(color='#334155')),
        decreasing=dict(marker_color='#ef4444'),
        increasing=dict(marker_color='#10b981'),
        totals=dict(marker_color='#6366f1'),
    ))
    fig.update_layout(title='Business Impact Waterfall (£)', height=460, **LAYOUT)
    fig.show()